# 02 - Silver checks: crashes and weather

Silver turns the crash and weather sources into typed, analysis-ready tables. Every rule comes from a finding of the proposal's EDA. This notebook checks that the rules did what the EDA said they should:

1. crashes: how many were kept, why the rest were dropped, and whether every community area is present;
2. weather: one FM-15 report per hour, decoded values in the EDA's ranges, and the UTC → Chicago shift across daylight saving.

In [1]:
import os

import pandas as pd
import pyspark.sql.functions as F
import yaml

# run from the repo root, where main.py, config/ and data/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import main

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)
spark = main.get_spark(config)

bronze_crashes = spark.read.parquet("datamart/bronze/crashes")
silver_crashes = spark.read.parquet("datamart/silver/crashes")
bronze_weather = spark.read.parquet("datamart/bronze/weather")
silver_weather = spark.read.parquet("datamart/silver/weather")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 08:20:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1 - Crashes

Silver keeps a crash only if it has a location inside one of the 77 community areas. The EDA found 0.72% of crashes with no usable latitude/longitude (null or 0).

In [2]:
lat = F.col("latitude").cast("double")
lon = F.col("longitude").cast("double")
no_location = lat.isNull() | (lat == 0) | lon.isNull() | (lon == 0)

bronze_rows = bronze_crashes.count()
without_location = bronze_crashes.filter(no_location).count()
silver_rows = silver_crashes.count()

print(f"bronze crashes: {bronze_rows:,}")
print(f"without location: {without_location:,} ({without_location / bronze_rows:.2%})")
print(f"outside every area: {bronze_rows - without_location - silver_rows:,}")
print(f"in silver: {silver_rows:,} ({silver_rows / bronze_rows:.2%} of bronze)")

bronze crashes: 527,518
without location: 3,668 (0.70%)
outside every area: 239
in silver: 523,611 (99.26% of bronze)


In [3]:
per_area = silver_crashes.groupBy("community_area").count().toPandas().set_index("community_area")["count"]
print("areas with crashes:", per_area.size, "| numbered", per_area.index.min(), "to", per_area.index.max())
print("fewest:", per_area.idxmin(), per_area.min(), "| most:", per_area.idxmax(), per_area.max())
per_area.sort_values(ascending=False).head(5)

areas with crashes: 77 | numbered 1 to 77
fewest: 47 456 | most: 25 25678


community_area
25    25678
28    24081
8     23250
32    16815
24    14974
Name: count, dtype: int64

In [4]:
# silver is typed: timestamps without time zone, coordinates as numbers, the area as an integer
silver_crashes.printSchema()

root
 |-- crash_record_id: string (nullable = true)
 |-- crash_date: timestamp_ntz (nullable = true)
 |-- date_police_notified: timestamp_ntz (nullable = true)
 |-- crash_type: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- _arrived_at: timestamp_ntz (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- run_id: string (nullable = true)



## 2 - Weather

Silver keeps the routine hourly report (FM-15), so there is normally one observation per hour, taken at :51. The EDA counted 48,217 FM-15 reports in the whole window; bronze holds the ones that arrived before go-live.

In [5]:
bronze_fm15 = bronze_weather.filter(F.trim("REPORT_TYPE") == "FM-15").count()
silver_rows = silver_weather.count()
utc_hours = silver_weather.select(F.date_trunc("hour", "observed_at_utc").alias("hour")).distinct().count()

print("FM-15 reports in bronze:", bronze_fm15)
print("rows in silver:         ", silver_rows)
print("distinct UTC hours:     ", utc_hours)

FM-15 reports in bronze: 42395
rows in silver:          42395
distinct UTC hours:      42390


In [6]:
# a few hours hold two FM-15 reports: which ones?
hour = F.date_trunc("hour", "observed_at_utc")
twice = silver_weather.groupBy(hour.alias("hour")).count().filter("count > 1").select("hour")
(silver_weather.join(twice, hour == twice["hour"])
    .select("observed_at_utc", "observed_at", "temp_c", "wind_ms", "precip_1h_mm")
    .orderBy("observed_at_utc").show())

+-------------------+-------------------+------+-------+------------+
|    observed_at_utc|        observed_at|temp_c|wind_ms|precip_1h_mm|
+-------------------+-------------------+------+-------+------------+
|2018-07-22 08:41:00|2018-07-22 03:41:00|  20.6|    5.7|        NULL|
|2018-07-22 08:51:00|2018-07-22 03:51:00|  20.6|    5.7|         0.0|
|2018-07-22 17:06:00|2018-07-22 12:06:00|  22.2|    7.7|        NULL|
|2018-07-22 17:51:00|2018-07-22 12:51:00|  22.8|    7.7|         0.0|
|2020-04-01 14:41:00|2020-04-01 09:41:00|   5.6|    2.6|        NULL|
|2020-04-01 14:51:00|2020-04-01 09:51:00|   5.6|    2.1|         0.0|
|2020-06-23 01:23:00|2020-06-22 20:23:00|  21.1|    5.1|        NULL|
|2020-06-23 01:51:00|2020-06-22 20:51:00|  21.1|    4.1|         6.4|
|2020-06-23 10:16:00|2020-06-23 05:16:00|  17.2|    4.6|        NULL|
|2020-06-23 10:51:00|2020-06-23 05:51:00|  17.2|    4.1|         0.0|
+-------------------+-------------------+------+-------+------------+



In [7]:
# the decoded values; the EDA's describe() over the whole window gave
# temp -30.6 to 37.8 C (mean 11.6), visibility 0 to 16,093 m, wind 0 to 17.0 m/s, precipitation 0 to 48.5 mm
columns = ["temp_c", "dew_point_c", "visibility_m", "wind_ms", "precip_1h_mm"]
silver_weather.select(columns).toPandas().describe().round(1)

,temp_c,dew_point_c,visibility_m,wind_ms,precip_1h_mm
count,42392.0,42387.0,42394.0,42392.0,41933.0
mean,11.4,4.9,14952.0,4.3,0.1
std,11.4,10.6,3135.6,2.3,1.0
min,-30.6,-35.6,0.0,0.0,0.0
25%,2.2,-2.8,16093.0,2.6,0.0
50%,11.1,4.4,16093.0,4.1,0.0
75%,21.1,14.4,16093.0,5.7,0.0
max,36.7,26.1,16093.0,17.0,48.5


In [8]:
# daylight saving: UTC minus Chicago time is 6 h in winter (CST) and 5 h in summer (CDT)
offset = (F.unix_timestamp("observed_at_utc") - F.unix_timestamp("observed_at")) / 3600
silver_weather.groupBy(offset.alias("utc_minus_chicago_h")).count().orderBy("utc_minus_chicago_h").show()

# the night the clocks went back (4 Nov 2018): 01:51 local happens twice
(silver_weather.filter(F.col("observed_at_utc").between("2018-11-04 05:00", "2018-11-04 09:00"))
    .select("observed_at_utc", "observed_at", "temp_c").orderBy("observed_at_utc").show())

+-------------------+-----+
|utc_minus_chicago_h|count|
+-------------------+-----+
|                5.0|28539|
|                6.0|13856|
+-------------------+-----+

+-------------------+-------------------+------+
|    observed_at_utc|        observed_at|temp_c|
+-------------------+-------------------+------+
|2018-11-04 05:51:00|2018-11-04 00:51:00|   8.3|
|2018-11-04 06:51:00|2018-11-04 01:51:00|   8.3|
|2018-11-04 07:51:00|2018-11-04 01:51:00|   7.8|
|2018-11-04 08:51:00|2018-11-04 02:51:00|   8.3|
+-------------------+-------------------+------+



**Findings:**
- **Crashes:** 527,518 in bronze; 3,668 (0.70%) have no location, as the EDA found (0.72% over the whole window), and 239 more lie outside every community area. **523,611 (99.26%) are in silver**, each with its area. All 77 areas have crashes, from 456 (area 47) to 25,678 (area 25).
- **Weather:** all 42,395 FM-15 reports that arrived before go-live are in silver, decoded. They cover 42,390 UTC hours: **5 hours have a second, off-schedule FM-15** (at :06, :16, :23 or :41, always followed by the routine :51 report, with no precipitation group). Silver keeps them as received; a "last reading before T" feature takes the :51 one.
- **Decoded values** fall in the EDA's ranges (temperature −30.6 to 36.7 °C, visibility up to 16,093 m, wind up to 17.0 m/s, precipitation up to 48.5 mm in an hour). A few values are missing (3 temperatures, 8 dew points, 1 visibility, 3 wind speeds), and precipitation is missing in 462 reports (1.1%).
- **Time zones:** UTC minus Chicago time is 6 h for 13,856 reports (winter) and 5 h for 28,539 (summer). On the fall-back night (4 Nov 2018), 01:51 local appears twice, an hour apart in real time; the daylight-saving rule for clock hours is applied in gold, not here.
- What this doesn't check: whether the **gold** features built from these tables use only what had arrived before each hour. That is what the gold point-in-time tests (tests/test_gold.py) check.